In [1]:
import pandas as pd
import re

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time

In [6]:
# Load the CSV
df = pd.read_csv("Ticket Template x Remote User Refresh - Current(Richardson).csv")

# Select columns
df_subset = df[["Username", "Phone Number", "Tracking Info", "Tech Assigned"]].copy()

# Extract valid UPS tracking numbers (starts with 1Z and 18 characters long)
def extract_1z_codes(text):
    text_str = str(text)
    matches = re.findall(r'\b1Z[0-9A-Z]{16}\b', text_str)
    result = {}
    for i, match in enumerate(matches):
        result[f"Tracking{i+1}"] = match
        result[f"Status{i+1}"] = ""  # Ready for status from your script

    # Add tracer flag if keyword appears (case-insensitive)
    result["Tracer"] = "Tracer" if "tracer" in text_str.lower() else ""

    return pd.Series(result)

# Apply the function
tracking_df = df_subset["Tracking Info"].apply(extract_1z_codes)
final_df = pd.concat([df_subset.drop("Tracking Info", axis=1), tracking_df], axis=1)

# Export for your tracking script
final_df.to_csv("RichardsonTracking.csv", index=False)

In [7]:
# Load the CSV
df = pd.read_csv("Ticket Template x Remote User Refresh - Current(Call-Ins and Orders).csv", encoding="ISO-8859-1")

# Select columns
df_subset = df[["Username", "Phone Number", "Tracking Info", "Tech Assigned"]].copy()

# Extract valid UPS tracking numbers (starts with 1Z and 18 characters long)
def extract_1z_codes(text):
    text_str = str(text)
    matches = re.findall(r'\b1Z[0-9A-Z]{16}\b', text_str)
    result = {}
    for i, match in enumerate(matches):
        result[f"Tracking{i+1}"] = match
        result[f"Status{i+1}"] = ""  # Ready for status from your script

    # Add tracer flag if keyword appears (case-insensitive)
    result["Tracer"] = "Tracer" if "tracer" in text_str.lower() else ""

    return pd.Series(result)

# Apply the function
tracking_df = df_subset["Tracking Info"].apply(extract_1z_codes)
final_df = pd.concat([df_subset.drop("Tracking Info", axis=1), tracking_df], axis=1)

# Export for your tracking script
final_df.to_csv("CallInTracking.csv", index=False)

In [8]:
# Load and normalize both CSVs
files = [
    "RichardsonTracking.csv",
    "CallInTracking.csv"
]

dataframes = []
for f in files:
    df = pd.read_csv(f)

    # Add any missing columns (Tracking4, Status4, etc.)
    for col in ["Tracking4", "Status4", "Tracking5", "Status5"]:
        if col not in df.columns:
            df[col] = ""

    dataframes.append(df)

# Stack them vertically
final_df = pd.concat(dataframes, axis=0, ignore_index=True)

# Ensure Tracer is last
tracer_col = final_df.pop("Tracer")
final_df["Tracer"] = tracer_col


# Export to unified CSV
final_df.to_csv("shipmentInput.csv", index=False)

In [9]:
%%time

# --- SETUP HEADLESS BROWSER ---
options = Options()
options.add_argument("--blink-settings=imagesEnabled=false")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# --- STATUS FETCH FUNCTION ---
def fetch_status(tracking_number):
    if pd.isna(tracking_number) or tracking_number.strip() == "":
        return ""

    url = f"https://www.ups.com/track?tracknum={tracking_number}&loc=en_US&requester=ST/trackdetails"
    driver.get(url)

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (By.XPATH, "//td[contains(text(),'Delivered') or contains(text(),'In Transit') or contains(text(),'Exception') or contains(text(),'On the Way')]")
            )
        )
        status_elements = driver.find_elements(
            By.XPATH,
            "//td[contains(text(),'Delivered') or contains(text(),'In Transit') or contains(text(),'Exception') or contains(text(),'On the Way')]"
        )

        # Prioritize status selection
        priority = ['Delivered', 'Exception', 'In Transit', 'On the Way']
        for p in priority:
            for element in status_elements:
                if p in element.text:
                    return p  # return only the status keyword, not the whole block

        return "Status not found"

    except Exception:
        return "Status not found"

# --- READ CSV AND PROCESS ---
df = pd.read_csv("shipmentInput.csv", encoding="latin1")  # or try encoding="cp1252"

# Loop through each row and apply status checks
for index, row in df.iterrows():
    for i in range(1, 6):
        track_col = f"Tracking{i}"
        stat_col = f"Status{i}"
        status = fetch_status(row.get(track_col))
        df.at[index, stat_col] = str(status)  # Explicitly cast to string

# --- FINALIZE ---
driver.quit()
df.to_csv("shipment-tracking.csv", index=False)

<timed exec>:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Delivered' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
<timed exec>:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Delivered' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
<timed exec>:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Delivered' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
<timed exec>:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
<timed exec>:52: FutureWa

CPU times: total: 344 ms
Wall time: 2min 16s
